In [ ]:
!pip install transformers torch biopython requests numpy scipy

In [ ]:
!curl -sSL https://securedna.github.io/ppa/deb/securedna-keyring.gpg | sudo tee /usr/share/keyrings/securedna-keyring.gpg > /dev/null
!echo "deb [signed-by=/usr/share/keyrings/securedna-keyring.gpg] https://securedna.github.io/ppa/deb ./" | sudo tee /etc/apt/sources.list.d/securedna.list > /dev/null
!sudo apt update && sudo apt install -y synthclient

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://securedna.github.io/ppa/deb ./ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 3s (1,189 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
107 packages can be upgraded. Run 'a

In [ ]:
# =============================================================================
# BIOSECURITY SCREENING PIPELINE
# =============================================================================
#
# Architecture (see diagram):
#   DNA input
#     └─ translate_dna()          — 6 reading frames (3 forward + 3 rev. comp.)
#         ├─ Layer 1: SecureDNA   — k-mer homology screen
#         ├─ Layer 2: ESM2        — protein embedding similarity vs toxin DB
#         │    └─ Layer 2.5       — optional trained classifier on top of embeddings
#         └─ combine_scores()     — final verdict: SAFE / SUSPICIOUS / DANGEROUS
#
# Modes (pass mode= to screen_sequence / run_batch):
#   "full"            — both layers, combined verdict
#   "securedna_only"  — Layer 1 only, no ESM2
#   "embeddings_only" — Layer 2 only, no SecureDNA
#
# Classifier (optional):
#   Train with train_classifier(), pass model= to screen_sequence / run_batch.
#   Without it, the system falls back to a fixed diff-score threshold.
#
# =============================================================================

# ── Cell 0: install & imports ─────────────────────────────────────────────────

# !pip install transformers torch biopython requests numpy scipy scikit-learn joblib

# SecureDNA synthclient (only needed for real Layer 1):
# !curl -sSL https://securedna.github.io/ppa/deb/securedna-keyring.gpg | sudo tee /usr/share/keyrings/securedna-keyring.gpg > /dev/null
# !echo "deb [signed-by=/usr/share/keyrings/securedna-keyring.gpg] https://securedna.github.io/ppa/deb ./" | sudo tee /etc/apt/sources.list.d/securedna.list > /dev/null
# !sudo apt update && sudo apt install -y synthclient

import os
import re
import csv
import json
import time
import shutil
import subprocess
import tempfile
import warnings
from dataclasses import dataclass, field, asdict
from typing import Optional

import joblib
import numpy as np
import requests
import torch
from Bio.Seq import Seq
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

print("✓ Imports OK")


✓ Imports OK


In [ ]:

# =============================================================================
# ── Cell 1: configuration ─────────────────────────────────────────────────────
# =============================================================================

#clone my folder to your drive

# from google.colab import drive

# drive.mount('/content/drive')
# DRIVE_DIR = "/content/drive/MyDrive/AIxBio Hackathon"
# os.makedirs(DRIVE_DIR, exist_ok=True)

# # Paths -----------------------------------------------------------------------
# TOXIN_DB_PATH   = f"{DRIVE_DIR}/toxin_embeddings_large.npz"
# NEUTRAL_DB_PATH = f"{DRIVE_DIR}/neutral_embeddings_large.npz"
# CRITICAL_DB_PATH = f"{DRIVE_DIR}/critical_toxins_embeddings.npz"
# MODEL_PATH      = f"{DRIVE_DIR}/classifier.joblib"
# REF_MEAN_PATH   = f"{DRIVE_DIR}/ref_mean.npy"

# UniProt fetch limits --------------------------------------------------------
MAX_TOXINS   = 25000   # reviewed UniProt has ~7 800 toxins; higher = no-op
MAX_NEUTRALS = 25000

# ESM2 model ------------------------------------------------------------------
# Options (smallest → largest, fastest → most accurate):
#   facebook/esm2_t6_8M_UR50D      8M params,   ~31 MB  — fast CPU baseline
#   facebook/esm2_t30_150M_UR50D  150M params,  ~591 MB  — good balance
#   facebook/esm2_t33_650M_UR50D  650M params,  ~2.5 GB  — best quality, needs GPU
ESM2_MODEL_NAME = "facebook/esm2_t33_650M_UR50D"

# SecureDNA -------------------------------------------------------------------
# SECUREDNA_CERT_PATH = os.getenv("SECUREDNA_CERT_PATH", "")
TURN_SECUREDNA_OFF  = False   # hard switch: set True to skip Layer 1 entirely

# Scoring thresholds (used when no classifier is provided) --------------------
DIFF_DANGEROUS  = 0.08
DIFF_SUSPICIOUS = 0.03
MODEL_DANGEROUS  = 0.50
MODEL_SUSPICIOUS = 0.20

In [ ]:
# =============================================================================
# ── Cell 2: data structures ───────────────────────────────────────────────────
# =============================================================================

@dataclass
class ScreeningResult:
    """All fields produced by screen_sequence() for a single DNA input."""

    # Raw input
    dna_sequence:     str
    protein_sequence: str = ""

    # Layer 1
    securedna_flagged: Optional[bool] = None   # True / False / None (error)
    securedna_raw:     dict = field(default_factory=dict)

    # Layer 2 (embedding similarity)
    S_toxin:         Optional[float] = None   # max cosine sim to toxin DB
    S_bg:            Optional[float] = None   # max cosine sim to background DB
    functional_score: Optional[float] = None  # S_toxin − S_bg
    top_matches:     list = field(default_factory=list)  # [(name, accession, score), ...]

    # Layer 2.5 (classifier, optional)
    model_score: Optional[float] = None   # calibrated probability ∈ [0, 1]

    # Final verdict
    combined_risk:  str = "UNKNOWN"        # SAFE / SUSPICIOUS / DANGEROUS / UNKNOWN
    combined_score: Optional[float] = None

    def to_dict(self) -> dict:
        return asdict(self)




In [ ]:
# =============================================================================
# ── Cell 3: DNA → protein translation ────────────────────────────────────────
# =============================================================================

def translate_dna(dna: str) -> str:
    """
    Translates DNA to the longest open reading frame (ORF) across all 6 frames:
    3 forward frames and 3 reverse-complement frames.

    Why 6 frames: a gene can be encoded on either strand, and the coding frame
    is unknown. Checking only the forward strand misses half of possible ORFs.

    Returns the longest ORF (truncated at first stop codon). Returns "" if
    all frames yield fewer than 5 amino acids.
    """
    dna = dna.upper().strip()

    # Build reverse complement manually (no external dependency)
    complement = str.maketrans("ACGT", "TGCA")
    rev_comp = dna.translate(complement)[::-1]

    best = ""

    for strand in [dna, rev_comp]:
        for frame in range(3):
            fragment = strand[frame:]
            fragment = fragment[:len(fragment) - len(fragment) % 3]
            protein = str(Seq(fragment).translate())

            # Truncate at first stop codon
            if "*" in protein:
                protein = protein[:protein.index("*")]

            if len(protein) > len(best):
                best = protein

    return best


In [ ]:
# =============================================================================
# ── Cell 4: Layer 1 — SecureDNA ──────────────────────────────────────────────
# =============================================================================
#
# SecureDNA screens DNA for sequences with significant homology to regulated
# biological agents (Select Agents, CBRN pathogens, etc.) using k-mer hashing.
# It requires a locally running synthclient binary and a valid certificate.
#
# When synthclient is unavailable (e.g. no registration yet), set TURN_SECUREDNA_OFF = True.
#
# Documentation: https://github.com/SecureDNA/SecureDNA/wiki/Synthclient-API
def _synthclient_available() -> bool:
    return shutil.which("synthclient") is not None


def _aa_to_dna_simple(aa_seq: str) -> str:
    table = {
        'A':'GCT','C':'TGT','D':'GAT','E':'GAA','F':'TTT','G':'GGT',
        'H':'CAT','I':'ATT','K':'AAA','L':'CTG','M':'ATG','N':'AAT',
        'P':'CCT','Q':'CAA','R':'CGT','S':'TCT','T':'ACT','V':'GTT',
        'W':'TGG','Y':'TAT','*':'TAA',
    }
    return "".join(table.get(aa.upper(), "GCT") for aa in aa_seq.strip())


def _screen_securedna_real(protein_seq: str) -> dict:
    t0 = time.time()
    try:
        dna = _aa_to_dna_simple(protein_seq)
        with tempfile.NamedTemporaryFile(mode="w", suffix=".fasta", delete=False) as f:
            f.write(f">query\n{dna}\n")
            fasta_path = f.name

        cmd = ["synthclient", fasta_path]
        if SECUREDNA_CERT_PATH and os.path.exists(SECUREDNA_CERT_PATH):
            cmd += ["--cert", SECUREDNA_CERT_PATH]

        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        os.unlink(fasta_path)

        elapsed = round(time.time() - t0, 3)

        if proc.returncode != 0:
            return {
                "flagged": None,
                "error": proc.stderr.strip(),
                "provider": "SecureDNA",
                "runtime_seconds": elapsed
            }

        data = json.loads(proc.stdout)
        flagged = any(r.get("hits") for r in data.get("results", []))

        return {
            "flagged": flagged,
            "hits": data.get("results", []),
            "provider": "SecureDNA",
            "runtime_seconds": elapsed
        }

    except Exception as e:
        return {
            "flagged": None,
            "error": str(e),
            "provider": "SecureDNA (exception)",
            "runtime_seconds": round(time.time() - t0, 3)
        }


def screen_layer1(protein_seq: str) -> dict:
    global TURN_SECUREDNA_OFF

    if TURN_SECUREDNA_OFF:
        return {
            "flagged": False,
            "provider": "SecureDNA (OFF)",
            "note": "TURN_SECUREDNA_OFF=True"
        }

    if not _synthclient_available():
        TURN_SECUREDNA_OFF = True
        return {
            "flagged": False,
            "provider": "SecureDNA (OFF)",
            "note": "synthclient not found → auto-disabled"
        }

    return _screen_securedna_real(protein_seq)

In [ ]:
# =============================================================================
# ── Cell 5: Layer 2 — ESM2 protein embeddings ────────────────────────────────
# =============================================================================
#
# ESM2 is a protein language model from Meta trained on ~250M UniProt sequences.
# It produces per-residue representations; we mean-pool them to get a single
# fixed-size vector per protein (shape: [hidden_dim]).
#
# Why mean pooling: each residue already encodes full sequence context via
# self-attention, so the mean captures global functional signal.
#
# Known limitation: active-site signal (~10 residues) can be diluted by bulk
# structural regions in long proteins. Future improvement: attention pooling.

_esm2_model     = None
_esm2_tokenizer = None


def load_esm2():
    """Loads ESM2 once into global cache. Subsequent calls are instant."""
    global _esm2_model, _esm2_tokenizer
    if _esm2_model is None:
        print(f"Loading ESM2 ({ESM2_MODEL_NAME})...")
        _esm2_tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL_NAME)
        _esm2_model     = AutoModel.from_pretrained(ESM2_MODEL_NAME)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _esm2_model = _esm2_model.to(device)
        _esm2_model.eval()
        print(f"✓ ESM2 on {device}")
    return _esm2_tokenizer, _esm2_model


def get_embedding(protein_seq: str) -> np.ndarray:
    """
    Returns a raw (un-normalized) ESM2 mean-pool embedding for a protein.
    Sequences longer than 1022 residues are truncated (model hard limit).
    Shape: [hidden_dim]  (320 for 8M, 640 for 150M, 1280 for 650M)
    """
    tokenizer, model = load_esm2()
    device = next(model.parameters()).device

    if len(protein_seq) > 1022:
        protein_seq = protein_seq[:1022]

    inputs = tokenizer(protein_seq, return_tensors="pt",
                       add_special_tokens=True).to(device)

    with torch.no_grad():
        out = model(**inputs)

    # Exclude [CLS] (position 0) and [EOS] (position -1) special tokens
    token_emb = out.last_hidden_state[0, 1:-1, :]   # [seq_len, hidden_dim]
    return token_emb.mean(dim=0).cpu().numpy()       # [hidden_dim]


def center_and_normalize(embeddings: np.ndarray, ref_mean: np.ndarray) -> np.ndarray:
    """
    Subtracts the global reference mean and L2-normalizes each row.

    Why this is necessary: ESM2 embeddings have a strong isotropic component
    shared by ALL proteins (a "protein-ness" bias). Without removing it, every
    protein looks nearly identical in cosine space (similarity → 1 everywhere).
    Subtracting the global mean (computed over both toxin + background DB)
    isolates the *differential* signal — what makes a toxin different from
    a random protein — which is what we actually want to measure.

    Args:
        embeddings: shape [N, D], raw ESM2 embeddings
        ref_mean:   shape [D], global mean over combined toxin + background DB
    Returns:
        shape [N, D], centered and L2-normalized
    """
    c     = embeddings - ref_mean
    norms = np.linalg.norm(c, axis=1, keepdims=True)
    return c / np.maximum(norms, 1e-8)


In [ ]:
# =============================================================================
# ── Cell 6: protein database — download & embed ───────────────────────────────
# =============================================================================

# Valid amino acid characters accepted by ESM2 tokenizer
_VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")


def fetch_proteins_paginated(
    query:        str,
    max_results:  int  = 25000,
    page_size:    int  = 500,
    save_path:    str  = "proteins.npz",
    force_rebuild: bool = False,
) -> dict:
    """
    Downloads proteins from UniProt Swiss-Prot and computes ESM2 embeddings.
    Saves raw (un-normalized) embeddings to an .npz file for reuse.

    Uses cursor-based pagination via the Link: <url>; rel="next" header, so
    it works for any result set size without hitting the 500-item API limit.

    Args:
        query:        UniProt REST query string (e.g. "keyword:KW-0800 AND reviewed:true")
        max_results:  stop after collecting this many proteins
        page_size:    items per API request (max 500 for UniProt)
        save_path:    where to write/read the .npz cache
        force_rebuild: ignore existing cache and rebuild from scratch

    Returns:
        dict with keys: names [N], accessions [N], embeddings [N, D]
    """
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

    # Load from cache if available
    if os.path.exists(save_path) and not force_rebuild:
        print(f"Loading from cache: {save_path}")
        d = np.load(save_path, allow_pickle=True)
        print(f"✓ {len(d['names'])} proteins loaded")
        return {"names": d["names"], "accessions": d["accessions"],
                "embeddings": d["embeddings"]}

    # ── Phase 1: fetch sequences from UniProt ─────────────────────────────────
    url         = "https://rest.uniprot.org/uniprotkb/search"
    params      = {"query": query, "format": "json",
                   "fields": "accession,protein_name,sequence", "size": page_size}
    next_url    = url
    next_params = params

    names, accessions, sequences = [], [], []
    seen = set()
    page = 0

    print(f"Fetching up to {max_results} proteins from UniProt...")

    while len(sequences) < max_results:
        # Fetch one page with one retry on transient errors
        for attempt in range(2):
            try:
                resp = requests.get(next_url, params=next_params, timeout=60)
                resp.raise_for_status()
                break
            except requests.exceptions.RequestException as e:
                if attempt == 0:
                    print(f"  Request failed ({e}), retrying in 10 s...")
                    time.sleep(10)
                else:
                    print(f"  Retry failed. Stopping fetch.")
                    break
        else:
            break

        results = resp.json().get("results", [])
        if not results:
            print("✓ UniProt returned no more results")
            break

        for entry in results:
            acc = entry.get("primaryAccession", "")
            if acc in seen:
                continue
            seen.add(acc)

            name = (entry.get("proteinDescription", {})
                        .get("recommendedName", {})
                        .get("fullName", {})
                        .get("value", "Unknown"))
            seq  = entry.get("sequence", {}).get("value", "")

            # Strip non-standard amino acids (e.g. X, U, B) that ESM2 rejects
            seq_clean = "".join(c for c in seq.upper() if c in _VALID_AA)

            if seq_clean and len(seq_clean) >= 10:
                names.append(name)
                accessions.append(acc)
                sequences.append(seq_clean)

        page += 1
        print(f"  Page {page}: total collected = {len(sequences)}", end="\r")

        # Follow pagination cursor from Link header
        link = resp.headers.get("Link", "")
        match = re.search(r'<([^>]+)>;\s*rel="next"', link)
        if not match:
            print(f"\n✓ No more pages after page {page}")
            break

        next_url    = match.group(1)
        next_params = None      # URL already contains all query parameters
        time.sleep(0.3)         # be polite to the API

        if len(sequences) >= max_results:
            print(f"\n✓ Reached target ({max_results})")
            break

    names      = names[:max_results]
    accessions = accessions[:max_results]
    sequences  = sequences[:max_results]

    # ── Phase 2: compute ESM2 embeddings ─────────────────────────────────────
    print(f"\nCollected {len(sequences)} sequences. Computing embeddings...")
    load_esm2()   # ensure model is loaded before the loop

    embeddings = []
    failed     = 0

    for i, seq in enumerate(sequences):
        if i % 100 == 0:
            print(f"  {i}/{len(sequences)} (failed: {failed})", end="\r")
        try:
            embeddings.append(get_embedding(seq))
        except Exception as e:
            if failed == 0:
                print(f"\n  First error at index {i}: {type(e).__name__}: {e}")
            embeddings.append(np.zeros(1280, dtype=np.float32))  # placeholder
            failed += 1

    emb_array = np.array(embeddings, dtype=np.float32)

    np.savez(save_path,
             names=np.array(names),
             accessions=np.array(accessions),
             embeddings=emb_array)

    print(f"\n✓ Saved {len(names)} proteins → {save_path}  (failed: {failed})")
    return {"names": np.array(names), "accessions": np.array(accessions),
            "embeddings": emb_array}


def compute_reference_mean(toxin_db: dict, bg_db: dict) -> np.ndarray:
    """
    Global mean over the combined toxin + background dataset.
    Must be computed from RAW (un-normalized) embeddings.
    Save this with np.save(REF_MEAN_PATH, ref_mean) so the same mean
    is used at training time and inference time.
    """
    all_emb = np.concatenate([toxin_db["embeddings"], bg_db["embeddings"]], axis=0)
    return all_emb.mean(axis=0)


In [ ]:
# =============================================================================
# ── Cell 7: Layer 2 — similarity scoring ─────────────────────────────────────
# =============================================================================

def score_dual_similarity(
    query_embedding: np.ndarray,
    toxin_db:        dict,
    bg_db:           dict,
    ref_mean:        np.ndarray,
    top_k:           int = 5,
) -> dict:
    """
    Computes the functional toxicity score for a single protein embedding.

    Approach:
        1. Center all embeddings (toxin DB, background DB, query) by subtracting
           the SAME global ref_mean.  Using the same reference is critical —
           different centers = different coordinate systems = meaningless cosine.
        2. L2-normalize to the unit sphere.
        3. Compute max cosine similarity to toxin DB  (S_toxin)
           and max cosine similarity to background DB (S_bg).
        4. diff = S_toxin − S_bg:
             > 0  → query is more toxin-like than background-like
             < 0  → query looks more like a random protein

    Args:
        query_embedding: shape [D], raw ESM2 embedding of the query protein
        toxin_db:        dict with "embeddings" key, shape [N_tox, D]
        bg_db:           dict with "embeddings" key, shape [N_bg, D]
        ref_mean:        shape [D], global mean (same object used at train time)
        top_k:           how many top toxin matches to return

    Returns:
        dict with: S_toxin, S_bg, diff, ratio, top_matches
    """
    tox_c   = center_and_normalize(toxin_db["embeddings"], ref_mean)
    bg_c    = center_and_normalize(bg_db["embeddings"],    ref_mean)
    query_c = center_and_normalize(query_embedding.reshape(1, -1), ref_mean)[0]

    s_tox = tox_c @ query_c
    s_bg  = bg_c  @ query_c

    max_tox = float(s_tox.max())
    max_bg  = float(s_bg.max())

    top_idx     = np.argsort(s_tox)[::-1][:top_k]
    top_matches = [(toxin_db["names"][i], toxin_db["accessions"][i], float(s_tox[i]))
                   for i in top_idx]

    return {
        "S_toxin":     max_tox,
        "S_bg":        max_bg,
        "diff":        max_tox - max_bg,
        "ratio":       max_tox / (max_bg + 1e-6),
        "top_matches": top_matches,
    }

In [ ]:
# =============================================================================
# ── Cell 8: Layer 2.5 — optional trained classifier ──────────────────────────
# =============================================================================
#
# A logistic regression trained directly on centered+normalized ESM2 embeddings.
# Calibrated with Platt scaling for well-calibrated probabilities.
#
# IMPORTANT: the embeddings fed to the classifier at training time must be
# processed with the SAME ref_mean used at inference time.  The ref_mean
# is saved alongside the model to enforce this.

def train_classifier(
    toxin_db:  dict,
    bg_db:     dict,
    ref_mean:  np.ndarray,
    test_size: float = 0.2,
) -> object:
    """
    Trains and calibrates a logistic regression classifier on ESM2 embeddings.

    Steps:
        1. Center+normalize embeddings using ref_mean  (same as inference)
        2. Train LogisticRegression on 80 % of data
        3. Apply Platt scaling (sigmoid calibration) on the held-out 20 %

    Why calibrate: raw logistic regression probabilities are often poorly
    calibrated (overconfident or underconfident).  Platt scaling fits a
    sigmoid on top to make predict_proba() outputs match true frequencies.

    Saves model to MODEL_PATH and ref_mean to REF_MEAN_PATH.
    Returns the calibrated classifier.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import roc_auc_score

    # Center+normalize with the SAME ref_mean that will be used at inference
    X_pos = center_and_normalize(toxin_db["embeddings"], ref_mean)
    X_neg = center_and_normalize(bg_db["embeddings"],    ref_mean)

    X = np.concatenate([X_pos, X_neg])
    y = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42)

    base = LogisticRegression(max_iter=1000, class_weight="balanced")
    base.fit(X_train, y_train)

    # Calibrate on the held-out set (cv="prefit" = model already trained)
    calibrated = CalibratedClassifierCV(estimator=base, method="sigmoid", cv="prefit")
    calibrated.fit(X_test, y_test)

    auc = roc_auc_score(y_test, calibrated.predict_proba(X_test)[:, 1])
    print(f"Classifier ROC-AUC (held-out): {auc:.4f}")

    os.makedirs(os.path.dirname(MODEL_PATH) or ".", exist_ok=True)
    joblib.dump(calibrated, MODEL_PATH)
    np.save(REF_MEAN_PATH, ref_mean)
    print(f"✓ Saved classifier → {MODEL_PATH}")
    print(f"✓ Saved ref_mean   → {REF_MEAN_PATH}")

    return calibrated


def load_classifier():
    """Loads the saved classifier and ref_mean from disk. Returns (model, ref_mean)."""
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"No classifier at {MODEL_PATH}. Run train_classifier() first.")
    model    = joblib.load(MODEL_PATH)
    ref_mean = np.load(REF_MEAN_PATH)
    print(f"✓ Classifier loaded from {MODEL_PATH}")
    return model, ref_mean


In [ ]:
train_classifier(toxin_db, bg_db, ref_mean)
model, ref_mean = load_classifier()

Classifier ROC-AUC (held-out): 0.9991
✓ Saved classifier → /content/drive/MyDrive/AIxBio Hackathon/classifier.joblib
✓ Saved ref_mean   → /content/drive/MyDrive/AIxBio Hackathon/ref_mean.npy
✓ Classifier loaded from /content/drive/MyDrive/AIxBio Hackathon/classifier.joblib


In [ ]:
# =============================================================================
# ── Cell 9: combine_scores — final decision logic ────────────────────────────
# =============================================================================

def combine_scores(
    securedna_flagged: Optional[bool],
    diff:              Optional[float],
    model_score:       Optional[float] = None,
) -> tuple[str, float]:
    """
    Determines the final risk verdict from all available signals.

    Decision table:
      SecureDNA = True  → DANGEROUS  (hard filter, overrides everything)
      SecureDNA = False → diff/model refine: at most SUSPICIOUS (not DANGEROUS)
      SecureDNA = None  → API error; treat diff/model as sole signal

    When classifier is available (model_score is not None):
        score = max(diff, model_score)     (union rule: either signal triggers)
    When classifier is absent:
        score = diff                        (embedding-only fallback)

    Thresholds are set in Cell 1 (DIFF_DANGEROUS, DIFF_SUSPICIOUS, etc.).
    Run calibrate_thresholds() after getting NIST results to tune them.

    Returns: (verdict: str, score: float)
    """
    diff  = diff        or 0.0
    score = model_score or diff
    if model_score is not None:
        score = max(diff, model_score)

    # SecureDNA hard flag — immediate DANGEROUS
    if securedna_flagged is True:
        return "DANGEROUS", score

    is_dangerous  = score >= (MODEL_DANGEROUS  if model_score is not None else DIFF_DANGEROUS)
    is_suspicious = score >= (MODEL_SUSPICIOUS if model_score is not None else DIFF_SUSPICIOUS)

    if securedna_flagged is None:
        # SecureDNA unavailable; use embedding/model alone
        if is_dangerous:  return "DANGEROUS",  score
        if is_suspicious: return "SUSPICIOUS", score
        return "SAFE", score

    # SecureDNA clear — downgrade DANGEROUS → SUSPICIOUS (don't fully trust embeddings)
    if is_dangerous or is_suspicious:
        return "SUSPICIOUS", score

    return "SAFE", score

In [ ]:
# =============================================================================
# ── Cell 10: screen_sequence — main single-sequence entry point ───────────────
# =============================================================================

VALID_MODES = ("full", "securedna_only", "embeddings_only")


def screen_sequence(
    dna_sequence: str,
    toxin_db:     dict,
    bg_db:        dict,
    ref_mean:     np.ndarray,
    model:        object         = None,
    mode:         str            = "full",
    verbose:      bool           = True,
) -> ScreeningResult:
    """
    Screens a single DNA sequence. The primary API for this pipeline.

    Args:
        dna_sequence: raw DNA string, any case, whitespace ignored
        toxin_db:     toxin embedding DB (from fetch_proteins_paginated)
        bg_db:        background embedding DB
        ref_mean:     global mean embedding (from compute_reference_mean or np.load)
        model:        optional trained classifier (from train_classifier or load_classifier)
        mode:         "full" | "securedna_only" | "embeddings_only"
        verbose:      print a detailed report for this sequence

    Returns:
        ScreeningResult with all relevant fields populated
    """
    if mode not in VALID_MODES:
        raise ValueError(f"mode must be one of {VALID_MODES}, got '{mode}'")

    dna_sequence = dna_sequence.upper().replace(" ", "").replace("\n", "")
    result = ScreeningResult(dna_sequence=dna_sequence)

    # ── Step 1: translate ─────────────────────────────────────────────────────
    result.protein_sequence = translate_dna(dna_sequence)

    if verbose:
        print(f"\n{'='*60}  [{mode}]")
        print(f"  DNA:     {dna_sequence[:60]}{'...' if len(dna_sequence)>60 else ''}")
        print(f"  Protein: {result.protein_sequence[:50]}{'...' if len(result.protein_sequence)>50 else ''}")
        print(f"  Lengths: {len(dna_sequence)} bp → {len(result.protein_sequence)} aa")

    if len(result.protein_sequence) < 5:
        if verbose:
            print("  ⚠ Protein too short (<5 aa) — skipped")
        result.combined_risk = "UNKNOWN"
        return result

    # ── Step 2: Layer 1 — SecureDNA ──────────────────────────────────────────
    if mode in ("full", "securedna_only"):
        sd = screen_layer1(result.protein_sequence)
        result.securedna_raw     = sd
        result.securedna_flagged = sd.get("flagged")

        if verbose:
            flag = "🚨 FLAGGED" if result.securedna_flagged else "✓ clean"
            print(f"  Layer 1 SecureDNA: {flag}  [{sd.get('provider','')}]")

        if mode == "securedna_only":
            result.combined_risk = "DANGEROUS" if result.securedna_flagged else "SAFE"
            return result

    # ── Step 3: Layer 2 — ESM2 embedding + similarity ────────────────────────
    if mode in ("full", "embeddings_only"):
        query_emb = get_embedding(result.protein_sequence)
        scores    = score_dual_similarity(query_emb, toxin_db, bg_db, ref_mean)

        result.S_toxin         = scores["S_toxin"]
        result.S_bg            = scores["S_bg"]
        result.functional_score = scores["diff"]
        result.top_matches     = scores["top_matches"]

        if verbose:
            print(f"  Layer 2 scores:  S_toxin={result.S_toxin:.4f}  "
                  f"S_bg={result.S_bg:.4f}  diff={result.functional_score:.4f}")
            print(f"  Top match: {result.top_matches[0][0][:50]}  "
                  f"({result.top_matches[0][2]:.4f})")

    # ── Step 4: Layer 2.5 — classifier (optional) ────────────────────────────
    if model is not None and mode in ("full", "embeddings_only"):
        # Feed centered+normalized embedding — same preprocessing as training
        x = center_and_normalize(query_emb.reshape(1, -1), ref_mean)
        result.model_score = float(model.predict_proba(x)[0, 1])

        if verbose:
            print(f"  Classifier score: {result.model_score:.4f}")

    # ── Step 5: combine into final verdict ────────────────────────────────────
    result.combined_risk, result.combined_score = combine_scores(
        result.securedna_flagged,
        result.functional_score,
        result.model_score,
    )

    if verbose:
        emoji = {"SAFE":"✅","SUSPICIOUS":"⚠️","DANGEROUS":"🚨","UNKNOWN":"❓"}
        print(f"  → {emoji.get(result.combined_risk,'?')} {result.combined_risk}  "
              f"(combined_score={result.combined_score:.4f})")

    return result


In [ ]:
# =============================================================================
# ── Cell 11: run_batch — screen many sequences with progress ──────────────────
# =============================================================================

def run_batch(
    sequences:    list[str],
    ids:          list[str],
    toxin_db:     dict,
    bg_db:        dict,
    ref_mean:     np.ndarray,
    model:        object = None,
    mode:         str    = "full",
    print_every:  int    = 10,
    verbose_each: bool   = False,
) -> list[ScreeningResult]:
    """
    Runs screen_sequence() over a list of sequences with a live progress bar.

    Args:
        sequences:    list of raw DNA strings
        ids:          list of sequence identifiers (for display only)
        toxin_db:     toxin embedding DB
        bg_db:        background embedding DB
        ref_mean:     global mean embedding
        model:        optional classifier
        mode:         "full" | "securedna_only" | "embeddings_only"
        print_every:  print progress every N sequences
        verbose_each: pass verbose=True to screen_sequence for every sequence

    Returns:
        list of ScreeningResult, one per input sequence
    """
    if mode not in VALID_MODES:
        raise ValueError(f"mode must be one of {VALID_MODES}")

    n       = len(sequences)
    results = []
    t0      = time.time()

    print(f"\n{'='*60}")
    print(f"  Batch screening  mode={mode}  n={n}")
    print(f"{'='*60}")

    for i, (seq_id, seq) in enumerate(zip(ids, sequences)):
        r = screen_sequence(seq, toxin_db, bg_db, ref_mean,
                            model=model, mode=mode, verbose=verbose_each)
        results.append(r)

        if (i + 1) % print_every == 0 or (i + 1) == n:
            elapsed = time.time() - t0
            per_seq = elapsed / (i + 1)
            eta     = per_seq * (n - i - 1)

            n_flagged = sum(1 for r in results
                            if r.combined_risk in ("SUSPICIOUS", "DANGEROUS"))
            print(f"  [{i+1:>5}/{n}]  elapsed={elapsed:6.1f}s  "
                  f"per_seq={per_seq:.2f}s  ETA={eta:5.1f}s  "
                  f"flagged={n_flagged}")

    print(f"\n✓ Done. Total: {time.time()-t0:.1f}s")
    return results



In [ ]:
!wget https://data.nist.gov/od/ds/mds2-3787/NIST_nucleic_acid_synthesis_screening_test_dataset.fasta
!wget https://data.nist.gov/od/ds/mds2-3787/NIST_nucleic_acid_syntheisis_screening_test_dataset_metadata.tsv

--2026-04-26 15:44:01--  https://data.nist.gov/od/ds/mds2-3787/NIST_nucleic_acid_synthesis_screening_test_dataset.fasta
Resolving data.nist.gov (data.nist.gov)... 172.65.90.24, 172.65.90.26, 172.65.90.25, ...
Connecting to data.nist.gov (data.nist.gov)|172.65.90.24|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nist-oar-cache.s3.amazonaws.com/prd/fst4/mds2-3787/NIST_nucleic_acid_synthesis_screening_test_dataset.fasta [following]
--2026-04-26 15:44:07--  https://nist-oar-cache.s3.amazonaws.com/prd/fst4/mds2-3787/NIST_nucleic_acid_synthesis_screening_test_dataset.fasta
Resolving nist-oar-cache.s3.amazonaws.com (nist-oar-cache.s3.amazonaws.com)... 54.231.170.113, 16.15.253.19, 16.15.252.236, ...
Connecting to nist-oar-cache.s3.amazonaws.com (nist-oar-cache.s3.amazonaws.com)|54.231.170.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 209712 (205K) [text/plain]
Saving to: ‘NIST_nucleic_acid_synthesis_screening_test_dataset

In [ ]:

# =============================================================================
# ── Cell 13: demo ─────────────────────────────────────────────────────────────
# =============================================================================

if __name__ == "__main__":

    toxin_db = np.load(TOXIN_DB_PATH)
    bg_db = np.load(NEUTRAL_DB_PATH)

    # # ── Build / load databases ────────────────────────────────────────────────
    # toxin_db = fetch_proteins_paginated(
    #     query      = "keyword:KW-0800 AND reviewed:true",
    #     max_results = MAX_TOXINS,
    #     save_path  = TOXIN_DB_PATH,
    #     force_rebuild = False,
    # )
    # bg_db = fetch_proteins_paginated(
    #     query = ("reviewed:true AND NOT keyword:KW-0800 "
    #              "AND NOT keyword:KW-1280 AND NOT keyword:KW-0964"),
    #     max_results = MAX_NEUTRALS,
    #     save_path  = NEUTRAL_DB_PATH,
    #     force_rebuild = False,
    # )

    ref_mean = compute_reference_mean(toxin_db, bg_db)

    # ── (Optional) train classifier ───────────────────────────────────────────
    # model = train_classifier(toxin_db, bg_db, ref_mean)
    # or load a pre-trained one:
    # model, ref_mean = load_classifier()
    model = None   # set to trained classifier to enable Layer 2.5

    # ── Test sequences ────────────────────────────────────────────────────────
    TEST = {
        # Real ricin A-chain fragment (should score high)
        "ricin_fragment": (
            "ATGGCTAGCATGACTGGTGGACAGCAAATGGGTCGCGGATCCGAGCTCGAATTCG"
            "GTACCGAGCTCGGATCCACTAGTAACGGCCGCCAGTGTGCTGGAATTCGCCCTT"
        ),
        # Synthetic benign signal peptide (should score low)
        "random_safe": (
            "ATGAAAGCCATTTTCGGATTTATCGCCCTGATCCTGGTTACCGCTGGACTGATCG"
            "CAACAGGTGCAGGTGGCAGCGGTGGCGGCAGTGGCAGTGGCAGCGGTGGCAGCG"
        ),
        # Very short fragment (should be UNKNOWN)
        "short_fragment": "ATGCGTACCGTTAAC",
    }

    for name, seq in TEST.items():
        print(f"\n--- {name} ---")
        screen_sequence(seq, toxin_db, bg_db, ref_mean,
                        model=model, mode="full", verbose=True)

NameError: name 'TOXIN_DB_PATH' is not defined

In [ ]:

# =============================================================================
# ── Cell 12: evaluation on NIST dataset ──────────────────────────────────────
# =============================================================================
#
# NIST dataset download:
# !wget https://data.nist.gov/od/ds/mds2-3787/NIST_nucleic_acid_synthesis_screening_test_dataset.fasta
# !wget https://data.nist.gov/od/ds/mds2-3787/NIST_nucleic_acid_syntheisis_screening_test_dataset_metadata.tsv
#
# Labels: TP = hazardous (SOC), TN = safe (non-SOC)


def load_nist_dataset(
    fasta_path:    str = "NIST_nucleic_acid_synthesis_screening_test_dataset.fasta",
    metadata_path: str = "NIST_nucleic_acid_syntheisis_screening_test_dataset_metadata.tsv",
) -> tuple[list[str], list[str], Optional[list[str]]]:
    """
    Loads the NIST biosecurity benchmark dataset.

    TSV columns: Label (row index), Designation (TP/TN), Name (matches FASTA id).

    Returns:
        ids:       FASTA record identifiers
        sequences: DNA sequences
        labels:    "TP" (hazardous) / "TN" (safe) per sequence, or None if TSV missing
    """
    from Bio import SeqIO

    ids, sequences = [], []
    for rec in SeqIO.parse(fasta_path, "fasta"):
        ids.append(rec.id)
        sequences.append(str(rec.seq))

    print(f"✓ FASTA: {len(sequences)} sequences")

    labels = None
    if os.path.exists(metadata_path):
        name_to_label = {}
        with open(metadata_path) as f:
            for row in csv.DictReader(f, delimiter="\t"):
                name  = row.get("Name", "").strip()
                desig = row.get("Designation", "").strip()   # TP or TN
                if name and desig:
                    name_to_label[name] = desig

        def _resolve(fasta_id):
            if fasta_id in name_to_label:
                return name_to_label[fasta_id]
            for tsv_name, d in name_to_label.items():
                if fasta_id in tsv_name or tsv_name in fasta_id:
                    return d
            return "unknown"

        labels = [_resolve(i) for i in ids]
        n_tp = labels.count("TP")
        n_tn = labels.count("TN")
        print(f"✓ Labels: TP={n_tp}  TN={n_tn}  unknown={labels.count('unknown')}")

    return ids, sequences, labels


def evaluate_on_nist(
    results:           list[ScreeningResult],
    labels:            list[str],
    layer2_threshold:  float = 0.05,
) -> dict:
    """
    Computes precision / recall / F1 / MCC metrics for each signal layer.

    Reports:
      Layer 1: SecureDNA alone
      Layer 2: embedding diff score alone (at layer2_threshold)
      Combined: final combined_risk verdict

    For biosecurity, FNR (miss rate) is the most critical metric:
    a missed dangerous sequence is far worse than a false alarm.

    Args:
        results:          output of run_batch()
        labels:           "TP" / "TN" ground truth per sequence
        layer2_threshold: functional_score cutoff for Layer 2 standalone decision

    Returns:
        dict with keys "layer1", "layer2", "combined", each a metrics dict
    """
    assert len(results) == len(labels)

    def _is_pos(label):
        return label.upper() == "TP"

    def _metrics(preds, labels):
        tp = fp = tn = fn = 0
        for p, l in zip(preds, labels):
            pos = _is_pos(l)
            if   p and pos:      tp += 1
            elif p and not pos:  fp += 1
            elif not p and pos:  fn += 1
            else:                tn += 1

        precision   = tp / (tp + fp)  if (tp + fp)  else 0.0
        recall      = tp / (tp + fn)  if (tp + fn)  else 0.0
        f1          = 2*precision*recall / (precision+recall) if (precision+recall) else 0.0
        specificity = tn / (tn + fp)  if (tn + fp)  else 0.0
        fpr         = fp / (fp + tn)  if (fp + tn)  else 0.0
        fnr         = fn / (fn + tp)  if (fn + tp)  else 0.0
        denom_mcc   = ((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) ** 0.5
        mcc         = (tp*tn - fp*fn) / denom_mcc if denom_mcc else 0.0

        return {"TP":tp,"FP":fp,"TN":tn,"FN":fn,
                "precision":precision,"recall":recall,"f1":f1,
                "specificity":specificity,"fpr":fpr,"fnr":fnr,"mcc":mcc}

    m = {
        "layer1":   _metrics([bool(r.securedna_flagged) for r in results], labels),
        "layer2":   _metrics([(r.functional_score or 0) >= layer2_threshold for r in results], labels),
        "combined": _metrics([r.combined_risk in ("SUSPICIOUS","DANGEROUS") for r in results], labels),
    }

    n_tp = sum(_is_pos(l) for l in labels)
    n_tn = len(labels) - n_tp

    print(f"\n{'='*70}")
    print(f"  NIST evaluation  —  {len(results)} sequences  (TP={n_tp}, TN={n_tn})")
    print(f"{'='*70}")
    header = f"  {'metric':<14}{'SecureDNA':>16}{'ESM2 diff':>16}{'Combined':>16}"
    print(header)
    print(f"  {'-'*14}{'-'*16}{'-'*16}{'-'*16}")
    for metric in ["precision","recall","f1","specificity","mcc","fpr","fnr"]:
        row = f"  {metric:<14}"
        for key in ("layer1","layer2","combined"):
            row += f"{m[key][metric]:>16.4f}"
        print(row)
    print(f"  {'-'*14}{'-'*16}{'-'*16}{'-'*16}")
    for count in ("TP","FP","TN","FN"):
        row = f"  {count:<14}"
        for key in ("layer1","layer2","combined"):
            row += f"{m[key][count]:>16}"
        print(row)
    print(f"{'='*70}")
    print(f"  ↑ FNR = miss rate (most critical for biosecurity)")

    return m


In [ ]:
#Testing on NIST dataset.
#THIS THING MUST USE GPU!!!!!

ids, seqs, labels = load_nist_dataset()

# Только SecureDNA
#results = run_batch(seqs, ids, toxin_db, bg_db, ref_mean,
#                    mode="securedna_only", print_every=50)
#metrics = evaluate_on_nist(results, labels, mode="securedna_only")

# Только embeddings
results = run_batch(seqs, ids, toxin_db, bg_db, ref_mean,
                    mode="embeddings_only", model=model, print_every=10)
metrics = evaluate_on_nist(results, labels)

# Полный прогон
#results = run_batch(seqs, ids, toxin_db, bg_db, ref_mean,
#                    mode="full", print_every=25)
#metrics = evaluate_on_nist(results, labels, mode="full")

FileNotFoundError: [Errno 2] No such file or directory: 'NIST_nucleic_acid_synthesis_screening_test_dataset.fasta'

In [ ]:
metrics = evaluate_on_nist(results, labels)

NameError: name 'results' is not defined

In [ ]:
print(type(model))
print(model is None)
results = run_batch(seqs[:5], ids[:5], toxin_db, bg_db, ref_mean,
                    mode="embeddings_only", model=model, verbose_each=True, print_every=1)
print([r.model_score for r in results])
print([r.functional_score for r in results])
print([r.combined_risk for r in results])


NameError: name 'model' is not defined

In [ ]:
# =============================================================================
# ── Cell X: protein FASTA → embed ────────────────────────────────────────────
# =============================================================================

import os
import numpy as np
import time

_VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")


def fetch_from_fasta_protein(
    fasta_path:    str,
    save_path:     str = "protein_embeddings.npz",
    force_rebuild: bool = False,
) -> dict:
    """
    Reads protein sequences from FASTA, cleans them, computes ESM2 embeddings.

    Args:
        fasta_path:   path to FASTA file with protein sequences
        save_path:    .npz cache
        force_rebuild: ignore cache

    Returns:
        dict with keys: names [N], sequences [N], embeddings [N, D]
    """

    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    # ── Phase 1: read FASTA ─────────────────────────────────────────────────
    names = []
    sequences = []

    with open(fasta_path, "r") as f:
        current_name = None
        current_seq = []

        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if current_name is not None:
                    sequences.append("".join(current_seq))
                    names.append(current_name)
                current_name = line[1:]
                current_seq = []
            else:
                current_seq.append(line)

        if current_name is not None:
            sequences.append("".join(current_seq))
            names.append(current_name)

    print(f"Loaded {len(sequences)} protein sequences")

    # ── Phase 2: clean sequences ────────────────────────────────────────────
    clean_seqs = []
    clean_names = []

    for i, seq in enumerate(sequences):
        if i % 100 == 0:
            print(f"  Cleaning {i}/{len(sequences)}", end="\r")

        seq = seq.upper()
        seq_clean = "".join(c for c in seq if c in _VALID_AA)

        if seq_clean and len(seq_clean) >= 10:
            clean_seqs.append(seq_clean)
            clean_names.append(names[i])

    print(f"\n✓ Kept {len(clean_seqs)} valid proteins")

    # ── Phase 3: embeddings ─────────────────────────────────────────────────
    load_esm2()

    embeddings = []
    failed = 0

    for i, seq in enumerate(clean_seqs):
        start = time.time()

        try:
            emb = get_embedding(seq)
            elapsed = time.time() - start

            embeddings.append(emb)

            print(
                f"[{i+1}/{len(clean_seqs)}] OK | len={len(seq)} | "
                f"time={elapsed:.3f}s | failed={failed}"
            )

        except Exception as e:
            elapsed = time.time() - start

            if failed == 0:
                print(f"\nFirst embedding error at {i}: {type(e).__name__}: {e}")

            embeddings.append(np.zeros(1280, dtype=np.float32))
            failed += 1

            print(
                f"[{i+1}/{len(clean_seqs)}] FAIL | len={len(seq)} | "
                f"time={elapsed:.3f}s | error={type(e).__name__}"
            )

    emb_array = np.array(embeddings, dtype=np.float32)

    # ── Save ────────────────────────────────────────────────────────────────
    np.savez(
        save_path,
        names=np.array(clean_names),
        sequences=np.array(clean_seqs),
        embeddings=emb_array,
    )

    print(f"\n✓ Saved {len(clean_seqs)} proteins → {save_path} (failed: {failed})")

    return {
        "names": np.array(clean_names),
        "sequences": np.array(clean_seqs),
        "embeddings": emb_array,
    }

db = fetch_from_fasta_protein(
    fasta_path="/content/content/protein.faa",
    save_path="/content/embeddings/generated_vaccinia.npz"
)

Loaded 223 protein sequences
  Cleaning 200/223
✓ Kept 223 valid proteins
Loading ESM2 (facebook/esm2_t33_650M_UR50D)...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ ESM2 on cpu
[1/223] OK | len=737 | time=20.754s | failed=0
[2/223] OK | len=493 | time=11.662s | failed=0
[3/223] OK | len=87 | time=3.083s | failed=0
[4/223] OK | len=263 | time=5.881s | failed=0
[5/223] OK | len=219 | time=4.965s | failed=0
[6/223] OK | len=177 | time=4.969s | failed=0
[7/223] OK | len=108 | time=2.647s | failed=0
[8/223] OK | len=574 | time=13.546s | failed=0
[9/223] OK | len=173 | time=7.669s | failed=0
[10/223] OK | len=259 | time=5.631s | failed=0
[11/223] OK | len=117 | time=2.851s | failed=0
[12/223] OK | len=552 | time=13.097s | failed=0
[13/223] OK | len=79 | time=2.520s | failed=0
[14/223] OK | len=512 | time=11.850s | failed=0
[15/223] OK | len=150 | time=4.464s | failed=0
[16/223] OK | len=74 | time=1.981s | failed=0
[17/223] OK | len=60 | time=1.689s | failed=0
[18/223] OK | len=283 | time=6.907s | failed=0
[19/223] OK | len=218 | time=5.068s | failed=0
[20/223] OK | len=90 | time=2.261s | failed=0
[21/223] OK | len=159 | time=3.678s | failed=0
[22/223]